# SecureScan — Automated Web Security Scanner

**IBM SkillsBuild Cyber Security 2026 · conducted by BharatCares**
**Project submission by:** Sakthi Vijay
**Category:** Web Application Security / Vulnerability Assessment · **Language:** Python 3.10+

## Objective
Build a small, beginner-friendly Python tool that evaluates the *basic security configuration* of a web endpoint by checking:

1. HTTPS availability and HTTP→HTTPS redirection
2. HTTP security response headers (CSP, HSTS, X-Content-Type-Options, X-Frame-Options, Referrer-Policy, Permissions-Policy)
3. Cookie security attributes (`Secure`, `HttpOnly`, `SameSite`)
4. Cross-Origin Resource Sharing (CORS) response configuration

For every finding the tool reports a **severity** (Informational, Low, Medium, High), the **evidence** it observed, **why it matters**, and a **remediation** recommendation. Results can be exported as **JSON** and **CSV**.

## Overview of this notebook
The notebook is organised in the order the scanner is built: setup → data models → URL validation → safe fetching → individual checks → orchestration → reporting → a **safe local demonstration server** → automated tests → conclusion.

> ### ⚠️ Authorized-use disclaimer
> SecureScan is intended for **authorized, defensive** security assessment only. Scan only systems you own or have **explicit written permission** to test. The built-in demonstration uses a **local test server on your own machine** and never contacts third-party sites. The tool refuses to scan non-local targets unless you explicitly confirm authorization.

> ### What this tool is — and is not
> SecureScan performs **configuration checks** on the HTTP responses it receives. A *missing* header or cookie flag is a **missing defensive control**, **not** a confirmed vulnerability, and this tool does **not** establish exploitability. It sends no exploit payloads, attempts no authentication bypass, and is **not** a penetration-testing tool. All findings are **preliminary** and need manual verification in context.

## 1. Problem statement and proposed solution

**Problem.** Many web applications are deployed with weak or missing security-related HTTP configuration: no HTTPS redirect, absent browser-protection headers, cookies without protective attributes, or over-permissive CORS. Beginners often do not know these controls exist, and manually inspecting responses with browser developer tools is slow and easy to get wrong. Full commercial scanners are heavy, and running aggressive tools against systems without permission is both unethical and often illegal.

**Proposed solution.** SecureScan is a lightweight, transparent, read-only scanner. It issues ordinary `GET` requests (plus one request that carries a harmless `Origin` header for the CORS check), analyses the response metadata, and explains each finding in plain language with documented severity rules. Response **bodies are never read or stored**, and cookie **values are never recorded** in reports.

## 2. Tools and libraries used

| Component | Purpose |
|---|---|
| **Python 3.10+** | Implementation language |
| **`requests`** (third-party, the only runtime dependency) | Sending HTTP/HTTPS requests with timeouts and TLS verification |
| `urllib.parse` | URL parsing and validation |
| `http.server` + `threading` | Local demonstration server (no external service needed) |
| `dataclasses` | Structured models for findings and scan results |
| `json`, `csv` | Report export |
| `re`, `ipaddress`, `socket`, `textwrap`, `pathlib`, `datetime`, `tempfile` | Parsing, loopback detection, formatting, files, timestamps |

Jupyter Notebook (`notebook`) is needed only to *open* this file; the scanner logic itself needs just `requests`.

## 3. Installation and setup

From the project folder, in a terminal:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate      macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
jupyter notebook SakthiVijay_SecureScan.ipynb
```

Then choose **Run → Run All Cells** (or run the cells top to bottom). Reports are written to an `output/` folder next to the notebook.

## 4. Imports and configuration
All settings live in one place. Timeouts are deliberately short so an unreachable host cannot hang the notebook.

In [1]:
import csv
import ipaddress
import json
import re
import tempfile
import textwrap
import threading
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin, urlsplit

import requests

TOOL_NAME = "SecureScan"
TOOL_VERSION = "1.0.0"

CONNECT_TIMEOUT = 5      # seconds to establish a connection
READ_TIMEOUT = 10        # seconds to wait for response headers
MAX_REDIRECTS = 5        # redirects followed manually, one hop at a time
MAX_URL_LENGTH = 2048
EVIDENCE_MAX_CHARS = 200 # long header values are truncated in reports
HSTS_MIN_MAX_AGE = 15_552_000  # 180 days; below this we report a Low finding

USER_AGENT = f"{TOOL_NAME}/{TOOL_VERSION} (educational configuration checker)"
OUTPUT_DIR = Path("output")

# Harmless origin used for the CORS check. ".invalid" is a reserved TLD that never resolves.
PROBE_ORIGIN = "https://cors-probe.invalid"

DISCLAIMER = (
    "SecureScan reports preliminary configuration findings only. A missing header or flag is a missing "
    "defensive control, not a confirmed vulnerability. Findings require manual verification. "
    "Use only on systems you are authorized to test."
)

print(f"{TOOL_NAME} v{TOOL_VERSION} loaded. requests {requests.__version__}")

SecureScan v1.0.0 loaded. requests 2.34.2

## 5. How findings are classified

Severity answers: *"how much does this missing or weak control matter if it is what it appears to be?"* — **not** *"is the site exploitable?"*

| Severity | Meaning |
|---|---|
| **Informational** | A control is present and looks reasonable, a check is not applicable, or a note for context |
| **Low** | Minor hardening gap or a weak setting; limited impact by itself |
| **Medium** | A significant defence-in-depth control is missing or clearly misconfigured |
| **High** | A configuration that, *if confirmed in context*, could expose users or data (e.g. no HTTPS at all, credentialed cross-origin reads from arbitrary origins) |

**Rule summary used by the code below**

| Check | Condition | Severity |
|---|---|---|
| HTTPS | HTTPS unreachable while HTTP responds | High |
| HTTPS | Certificate cannot be validated | Medium |
| HTTPS | HTTP served without redirecting to HTTPS (HTTPS available) | Medium |
| CSP | Header missing | Medium |
| CSP | Report-only only; no `script-src`/`default-src`; `unsafe-inline`/`unsafe-eval`; scheme-wide `http:`/`https:` sources | Low |
| CSP | `*` or `data:` allowed as script source | Medium |
| HSTS (HTTPS only) | Missing, or `max-age=0` | Medium |
| HSTS (HTTPS only) | `max-age` invalid or below 180 days | Low |
| HSTS | Response served over HTTP (header ignored by browsers) | Informational |
| X-Content-Type-Options | Missing or not `nosniff` | Low |
| X-Frame-Options | Missing and no CSP `frame-ancestors`; or unrecognised value | Low |
| Referrer-Policy | `unsafe-url` | Low |
| Referrer-Policy / Permissions-Policy | Missing or present | Informational |
| Cookie `Secure` | Missing on HTTPS, session-like name | Medium |
| Cookie `Secure` | Missing on HTTPS (other name), or response was over HTTP | Low |
| Cookie `HttpOnly` | Missing, session-like name | Medium |
| Cookie `HttpOnly` | Missing, other name (may be intentionally script-readable) | Low |
| Cookie `SameSite` | Missing or invalid value | Low |
| Cookie `SameSite` | `None` without `Secure` | Medium |
| CORS | Origin reflected **with** credentials allowed | High |
| CORS | `null` origin allowed with credentials | High |
| CORS | `null` origin allowed, or origin reflected without credentials | Medium / Low |
| CORS | `*` together with `Allow-Credentials: true` | Medium |
| CORS | `*` alone, specific origin, or no CORS headers | Informational |

"Session-like" cookie names are detected heuristically (names containing `sess`, `sid`, `auth`, `token`, `jwt`, `login`). This is a *hint*, not proof of what the cookie is used for.

## 6. Data models

In [2]:
SEVERITIES = ["Informational", "Low", "Medium", "High"]
SEVERITY_RANK = {name: index for index, name in enumerate(SEVERITIES)}


@dataclass
class Finding:
    """One observation produced by a check."""
    check: str        # category, e.g. "Security Header"
    item: str         # specific item, e.g. "Content-Security-Policy"
    severity: str     # one of SEVERITIES
    title: str        # short human-readable summary
    evidence: str     # what the scanner actually observed
    explanation: str  # why it matters (and its limits)
    remediation: str  # what to do about it

    def __post_init__(self):
        if self.severity not in SEVERITY_RANK:
            raise ValueError(f"Unknown severity: {self.severity}")


@dataclass
class ResponseSnapshot:
    """The parts of an HTTP response the checks need. Bodies and cookie values are never kept."""
    requested_url: str
    final_url: str = ""
    status_code: Optional[int] = None
    headers: dict = field(default_factory=dict)       # lower-case name -> value (Set-Cookie excluded)
    set_cookies: list = field(default_factory=list)   # raw Set-Cookie lines (values redacted at parse time)
    redirect_chain: list = field(default_factory=list)  # [(status, from_url, to_url), ...]
    notes: list = field(default_factory=list)
    error: Optional[str] = None
    error_type: Optional[str] = None

    @property
    def ok(self) -> bool:
        return self.error is None and self.status_code is not None


@dataclass
class ScanResult:
    """Everything produced by scanning one target."""
    target: str
    started_utc: str
    final_url: str = ""
    status_code: Optional[int] = None
    https_available: Optional[bool] = None
    redirect_chain: list = field(default_factory=list)
    notes: list = field(default_factory=list)
    error: Optional[str] = None
    findings: list = field(default_factory=list)

    def counts(self) -> dict:
        totals = {name: 0 for name in SEVERITIES}
        for item in self.findings:
            totals[item.severity] += 1
        return totals

    def highest(self) -> str:
        if not self.findings:
            return "None"
        return max((f.severity for f in self.findings), key=lambda s: SEVERITY_RANK[s])


def trunc(text, limit=EVIDENCE_MAX_CHARS):
    """Shorten long evidence strings so reports stay readable."""
    text = str(text)
    return text if len(text) <= limit else text[: limit - 3] + "..."


print("Models defined:", Finding.__name__, ResponseSnapshot.__name__, ScanResult.__name__)

Models defined: Finding ResponseSnapshot ScanResult

## 7. Input handling and URL validation
`validate_url` is the gatekeeper for every scan. It:
- rejects empty input, whitespace/control characters, over-long URLs, and URLs with embedded credentials (`user:pass@host`);
- allows **only `http` and `https`** (so `file://`, `ftp://`, `javascript:` and similar are refused);
- checks the host name and port syntax;
- assumes `https://` when the user omits the scheme, and drops any `#fragment`.

`is_local_host` recognises loopback targets (`localhost`, `127.0.0.0/8`, `::1`) — the only targets scanned without an explicit authorization confirmation.

In [3]:
class InvalidURLError(ValueError):
    """Raised when user input is not an acceptable http(s) URL."""


_HOST_LABEL = re.compile(r"^[A-Za-z0-9]([A-Za-z0-9-]{0,61}[A-Za-z0-9])?$")


def _valid_hostname(host: str) -> bool:
    try:
        ipaddress.ip_address(host)
        return True
    except ValueError:
        pass
    try:
        ascii_host = host.encode("idna").decode("ascii")  # supports internationalised names
    except UnicodeError:
        return False
    if len(ascii_host) > 253:
        return False
    return all(_HOST_LABEL.match(label) for label in ascii_host.rstrip(".").split("."))


def validate_url(raw) -> str:
    """Return a cleaned absolute http(s) URL or raise InvalidURLError."""
    if raw is None or not str(raw).strip():
        raise InvalidURLError("The URL is empty.")
    text = str(raw).strip()
    if len(text) > MAX_URL_LENGTH:
        raise InvalidURLError(f"The URL is longer than {MAX_URL_LENGTH} characters.")
    if any(ch.isspace() or ord(ch) < 32 for ch in text):
        raise InvalidURLError("The URL contains whitespace or control characters.")
    if "://" not in text:
        text = "https://" + text  # convenience: 'example.com' -> 'https://example.com'
    parts = urlsplit(text)
    if parts.scheme.lower() not in ("http", "https"):
        raise InvalidURLError(f"Unsupported scheme '{parts.scheme}'. Only http and https are allowed.")
    if parts.username or parts.password:
        raise InvalidURLError("URLs with embedded credentials are not accepted.")
    try:
        _ = parts.port  # raises ValueError for non-numeric or out-of-range ports
    except ValueError:
        raise InvalidURLError("The port number is invalid.") from None
    host = parts.hostname
    if not host:
        raise InvalidURLError("The URL has no host name.")
    if not _valid_hostname(host):
        raise InvalidURLError(f"'{host}' is not a valid host name.")
    return parts._replace(fragment="").geturl()


def is_local_host(host) -> bool:
    """True for loopback targets such as localhost, 127.0.0.1 and ::1."""
    if not host:
        return False
    if host.lower() == "localhost":
        return True
    try:
        return ipaddress.ip_address(host).is_loopback
    except ValueError:
        return False


for sample in ["example.com", "http://127.0.0.1:8000/a#frag", "ftp://example.com", "javascript:alert(1)", "", "https://user:pw@example.com"]:
    try:
        print(f"{sample!r:42} -> OK: {validate_url(sample)}")
    except InvalidURLError as err:
        print(f"{sample!r:42} -> rejected: {err}")

'example.com'                              -> OK: https://example.com
'http://127.0.0.1:8000/a#frag'             -> OK: http://127.0.0.1:8000/a
'ftp://example.com'                        -> rejected: Unsupported scheme 'ftp'. Only http and https are allowed.
'javascript:alert(1)'                      -> rejected: The port number is invalid.
''                                         -> rejected: The URL is empty.
'https://user:pw@example.com'              -> rejected: URLs with embedded credentials are not accepted.

## 8. Safe fetching with cautious redirect handling
`fetch_snapshot` sends a plain `GET` and returns a `ResponseSnapshot`. Design decisions:

- **Timeouts** on connect and read; **TLS certificates are verified** (never disabled).
- **Redirects are followed manually**, one hop at a time, up to `MAX_REDIRECTS`.
- A redirect is **not followed** if it leaves the original host (a leading `www.` is treated as the same site), changes to a non-http(s) scheme, loops, or exceeds the limit. In those cases the scan analyses the redirect response itself and records a note — so the scanner never wanders onto a site you did not choose.
- `stream=True` means the **response body is never downloaded**.
- **Cookie values** are never stored; they are redacted when parsed (next sections).
- Every network failure (DNS, refused, timeout, TLS, proxy, other) is **caught and classified** — the function never raises for an unreachable site.

In [4]:
REDIRECT_CODES = (301, 302, 303, 307, 308)

ERROR_MESSAGES = {
    "dns": "DNS lookup failed (the host name could not be resolved).",
    "refused": "Connection refused (nothing is listening on that host/port).",
    "timeout": "The connection or response timed out.",
    "tls_cert": "TLS certificate could not be validated (expired, self-signed, wrong host name or unknown issuer).",
    "tls": "TLS handshake failed (the port may not speak TLS, or protocols/ciphers are incompatible).",
    "proxy": "A network proxy refused or failed the connection.",
    "connection": "A network connection error occurred.",
    "request": "The HTTP request could not be completed.",
}
_DNS_HINTS = ("name or service not known", "getaddrinfo failed", "nodename nor servname",
              "temporary failure in name resolution", "failed to resolve", "name resolution")


def _same_site_host(a, b) -> bool:
    normalise = lambda h: (h or "").lower().removeprefix("www.")
    return normalise(a) == normalise(b)


def _record(snap: ResponseSnapshot, resp, url: str) -> None:
    """Copy only the metadata we need from a response."""
    snap.final_url = url
    snap.status_code = resp.status_code
    snap.headers = {k.lower(): v for k, v in resp.headers.items() if k.lower() != "set-cookie"}
    try:
        snap.set_cookies = list(resp.raw.headers.getlist("Set-Cookie"))
    except Exception:  # fall back gracefully if the raw header API is unavailable
        snap.set_cookies = []


def _fail(snap: ResponseSnapshot, kind: str) -> ResponseSnapshot:
    snap.error_type = kind
    snap.error = ERROR_MESSAGES[kind]
    return snap


def fetch_snapshot(url, extra_headers=None, allow_cross_host=False, timeout=None) -> ResponseSnapshot:
    """GET `url`, follow safe same-host redirects, and return a ResponseSnapshot (never raises on network errors)."""
    timeout = timeout or (CONNECT_TIMEOUT, READ_TIMEOUT)
    snap = ResponseSnapshot(requested_url=url)
    request_headers = {"User-Agent": USER_AGENT, "Accept": "*/*"}
    if extra_headers:
        request_headers.update(extra_headers)

    origin_host = urlsplit(url).hostname
    current, visited = url, {url}
    try:
        with requests.Session() as session:
            for hop in range(MAX_REDIRECTS + 1):
                resp = session.get(current, headers=request_headers, timeout=timeout,
                                   allow_redirects=False, stream=True)
                try:
                    location = resp.headers.get("Location")
                    if resp.status_code in REDIRECT_CODES and location:
                        target = urljoin(current, location)
                        target_parts = urlsplit(target)
                        snap.redirect_chain.append((resp.status_code, current, target))
                        stop = None
                        if target_parts.scheme.lower() not in ("http", "https"):
                            stop = f"Redirect to a non-http(s) scheme was not followed: {trunc(target, 80)}"
                        elif not allow_cross_host and not _same_site_host(target_parts.hostname, origin_host):
                            stop = (f"Redirect to a different host ({target_parts.hostname}) was NOT followed; "
                                    "results describe the redirect response itself.")
                        elif target in visited:
                            stop = "A redirect loop was detected; following stopped."
                        elif hop == MAX_REDIRECTS:
                            stop = f"Stopped after {MAX_REDIRECTS} redirects."
                        if stop:
                            snap.notes.append(stop)
                            _record(snap, resp, current)
                            return snap
                        visited.add(target)
                        current = target
                        continue
                    _record(snap, resp, current)
                    return snap
                finally:
                    resp.close()
    except requests.exceptions.SSLError as exc:
        return _fail(snap, "tls_cert" if "certificate verify failed" in str(exc).lower() else "tls")
    except requests.exceptions.Timeout:
        return _fail(snap, "timeout")
    except requests.exceptions.ProxyError:
        return _fail(snap, "proxy")
    except requests.exceptions.ConnectionError as exc:
        message = str(exc).lower()
        if any(hint in message for hint in _DNS_HINTS):
            return _fail(snap, "dns")
        if "refused" in message:
            return _fail(snap, "refused")
        return _fail(snap, "connection")
    except requests.exceptions.RequestException:
        return _fail(snap, "request")
    return snap


print("fetch_snapshot ready. Timeouts (connect, read):", (CONNECT_TIMEOUT, READ_TIMEOUT))

fetch_snapshot ready. Timeouts (connect, read): (5, 10)

## 9. HTTPS availability check
The check answers three questions: *Is HTTPS available? Is the certificate acceptable? Does plain HTTP redirect to HTTPS?*

- For loopback targets using `http://` the check is skipped (a local demo server is plain HTTP by design) and reported as Informational.
- For non-standard ports the HTTPS/HTTP counterpart probes are skipped, because a different port is a different service.
- The extra probes are ordinary `GET` requests to the same host on ports 443/80.

In [5]:
def _bracket(host: str) -> str:
    return f"[{host}]" if ":" in host else host


def check_https(requested_url: str, primary: ResponseSnapshot, timeout=None):
    """Return (findings, https_available) for the target. `primary` is the already-fetched response."""
    parts = urlsplit(requested_url)
    host, scheme = parts.hostname, parts.scheme.lower()
    path_query = (parts.path or "/") + (f"?{parts.query}" if parts.query else "")
    final_https = primary.final_url.lower().startswith("https://")
    findings = []

    if is_local_host(host) and scheme == "http":
        findings.append(Finding(
            "HTTPS", "Transport", "Informational", "Local test server uses plain HTTP (HTTPS check skipped)",
            f"Target {host} is a loopback address served over http://.",
            "Local demonstration servers are normally plain HTTP. HSTS and the cookie Secure flag cannot be "
            "meaningfully evaluated over HTTP.",
            "No action needed for a local demo. Production sites must use HTTPS."))
        return findings, None

    standard_port = parts.port in (None, 80, 443)
    netloc = _bracket(host)

    # --- Is HTTPS available? ---
    https_ok, https_snap = None, None
    if final_https:
        https_ok = True
    elif standard_port:
        https_snap = fetch_snapshot(f"https://{netloc}{path_query}", timeout=timeout)
        https_ok = https_snap.ok

    if https_ok:
        findings.append(Finding(
            "HTTPS", "Availability", "Informational", "HTTPS is available",
            f"A valid TLS connection was made (final URL: {trunc(primary.final_url if final_https else https_snap.final_url, 120)}).",
            "Encrypted transport protects confidentiality and integrity of traffic in transit.",
            "No action needed. Keep certificates renewed and use modern TLS versions."))
    elif https_ok is False and https_snap.error_type == "tls_cert":
        findings.append(Finding(
            "HTTPS", "Certificate", "Medium", "HTTPS endpoint certificate could not be validated",
            f"{https_snap.error}",
            "Browsers show warnings for certificates that are expired, self-signed, issued for another host name "
            "or from an untrusted issuer. The cause may be a test certificate; this tool cannot tell.",
            "Install a certificate from a trusted CA (e.g. Let's Encrypt) that matches the host name, and automate renewal."))
    elif https_ok is False:
        findings.append(Finding(
            "HTTPS", "Availability", "High", "HTTPS does not appear to be available",
            f"Plain HTTP responded, but the HTTPS attempt failed: {https_snap.error}",
            "Without HTTPS, traffic can be read or modified by anyone on the network path. This is a configuration "
            "observation from one probe; a firewall or a temporary fault could also explain it.",
            "Enable HTTPS with a trusted certificate, then redirect all HTTP traffic to HTTPS."))
    else:  # non-standard port, response was plain HTTP
        findings.append(Finding(
            "HTTPS", "Availability", "Medium", "Response was served over HTTP on a non-standard port",
            f"Final URL: {trunc(primary.final_url, 120)}. The HTTPS counterpart is not probed on non-standard ports.",
            "Cleartext HTTP exposes traffic to interception. HTTPS may be offered on another port that was not tested.",
            "Serve the application over HTTPS, or confirm the service is only reachable on a trusted network."))

    # --- Does plain HTTP redirect to HTTPS? ---
    if standard_port:
        http_snap = primary if scheme == "http" else fetch_snapshot(f"http://{netloc}{path_query}", timeout=timeout)
        if http_snap.ok:
            chain_to_https = any(hop[2].lower().startswith("https://") for hop in http_snap.redirect_chain)
            if http_snap.final_url.lower().startswith("https://") or chain_to_https:
                findings.append(Finding(
                    "HTTPS", "HTTP redirect", "Informational", "HTTP requests are redirected to HTTPS",
                    f"http:// request led to a redirect toward https:// ({len(http_snap.redirect_chain)} hop(s) observed).",
                    "Redirecting plain HTTP to HTTPS moves visitors to the encrypted site automatically.",
                    "No action needed. Consider adding HSTS so browsers skip the HTTP step entirely."))
            elif https_ok:
                findings.append(Finding(
                    "HTTPS", "HTTP redirect", "Medium", "HTTP is served without redirecting to HTTPS",
                    f"http:// responded with status {http_snap.status_code} and no redirect to https://.",
                    "Visitors who type or follow an http:// link stay on an unencrypted connection even though HTTPS exists.",
                    "Redirect all HTTP requests to HTTPS (301/308) and enable HSTS."))
        elif scheme == "https":
            findings.append(Finding(
                "HTTPS", "HTTP redirect", "Informational", "Plain HTTP port was not reachable",
                f"http:// probe failed: {http_snap.error}",
                "Closing port 80 is acceptable, though visitors typing the bare domain may then fail to connect.",
                "Consider answering on port 80 only to redirect to HTTPS."))
    return findings, https_ok


print("check_https defined.")

check_https defined.

## 10. HTTP security header analysis
Each function inspects one header and returns `Finding` objects. A short reminder of what each header does:

| Header | What it does | Why it matters |
|---|---|---|
| **Content-Security-Policy** | Tells the browser which sources of scripts, frames, images, etc. are allowed | A strong policy limits the damage of injected script (XSS) |
| **Strict-Transport-Security** | Tells browsers to use HTTPS only for this site for `max-age` seconds | Prevents downgrade and cookie-sniffing on first-hop HTTP |
| **X-Content-Type-Options** | `nosniff` stops browsers guessing a file's type | Reduces MIME-confusion attacks |
| **X-Frame-Options** | Controls whether the page may be framed | Mitigates clickjacking (CSP `frame-ancestors` is the modern equivalent) |
| **Referrer-Policy** | Controls how much URL information is sent in the `Referer` header | Limits leakage of URLs to other sites |
| **Permissions-Policy** | Restricts browser features (camera, geolocation, …) | Reduces exposure if a page or embedded frame is abused |

Remember: a **missing** header is a missing *control*, not proof of an exploitable flaw.

In [6]:
def _header(snap: ResponseSnapshot, name: str):
    return snap.headers.get(name.lower())


def _csp_directives(value: str) -> dict:
    directives = {}
    for chunk in value.split(";"):
        tokens = chunk.strip().split()
        if tokens:
            directives.setdefault(tokens[0].lower(), tokens[1:])  # first occurrence wins, as in browsers
    return directives


def check_csp(snap: ResponseSnapshot):
    name = "Content-Security-Policy"
    value, report_only = _header(snap, name), _header(snap, name + "-Report-Only")
    why = ("CSP restricts where scripts and other resources may load from, limiting the impact of injected script. "
           "Its absence is a missing defence-in-depth control, not proof of an XSS vulnerability.")
    fix = ("Define a CSP, starting with e.g. \"default-src 'self'; object-src 'none'; base-uri 'self'; frame-ancestors 'self'\", "
           "test in report-only mode first, then enforce and tighten.")
    if not value:
        if report_only:
            return [Finding("Security Header", name, "Low", "CSP is report-only (not enforced)",
                            f"Content-Security-Policy-Report-Only: {trunc(report_only)}", why, fix)]
        return [Finding("Security Header", name, "Medium", "Content-Security-Policy header not present",
                        "Header absent from the response.", why, fix)]

    directives = _csp_directives(value)
    script_src = directives.get("script-src") or directives.get("default-src")
    issues = []  # (severity, text)
    if script_src is None:
        issues.append(("Low", "no script-src or default-src, so script sources are not restricted"))
    else:
        tokens = [t.lower() for t in script_src]
        has_nonce_or_hash = any(t.startswith(("'nonce-", "'sha256-", "'sha384-", "'sha512-")) or t == "'strict-dynamic'" for t in tokens)
        if "*" in tokens:
            issues.append(("Medium", "wildcard (*) allowed as a script source"))
        if "data:" in tokens:
            issues.append(("Medium", "data: URIs allowed as a script source"))
        if "'unsafe-inline'" in tokens and not has_nonce_or_hash:
            issues.append(("Low", "'unsafe-inline' allowed for scripts"))
        if "'unsafe-eval'" in tokens:
            issues.append(("Low", "'unsafe-eval' allowed for scripts"))
        if any(t in ("http:", "https:") for t in tokens):
            issues.append(("Low", "scheme-wide http:/https: script sources allowed"))
    if not issues:
        return [Finding("Security Header", name, "Informational", "CSP present; no obvious weaknesses found by simple checks",
                        f"{name}: {trunc(value)}",
                        "The checks here are shallow: they cannot judge whether the policy is complete or effective for this application.",
                        "Review the policy manually; keep it as strict as the application allows.")]
    severity = max((s for s, _ in issues), key=lambda s: SEVERITY_RANK[s])
    return [Finding("Security Header", name, severity, "CSP present but weak in places",
                    f"{name}: {trunc(value)} | Observations: " + "; ".join(text for _, text in issues),
                    "Permissive script sources weaken the protection CSP is meant to provide. Whether this is exploitable depends on the application.",
                    "Remove wildcards, data:, 'unsafe-inline' and 'unsafe-eval'; prefer nonces or hashes for inline scripts.")]


def check_hsts(snap: ResponseSnapshot):
    name = "Strict-Transport-Security"
    value = _header(snap, name)
    over_https = snap.final_url.lower().startswith("https://")
    why = "HSTS makes browsers use HTTPS only for the site, reducing exposure to downgrade and interception on the first request."
    fix = "Send e.g. 'Strict-Transport-Security: max-age=31536000; includeSubDomains' over HTTPS once all subdomains support HTTPS."
    if not over_https:
        return [Finding("Security Header", name, "Informational", "HSTS not applicable: response was served over HTTP",
                        f"Final URL scheme is http. Header {'present' if value else 'absent'}.",
                        "Browsers ignore HSTS received over plain HTTP, so it cannot be assessed here.",
                        "Serve the site over HTTPS, then add HSTS.")]
    if not value:
        return [Finding("Security Header", name, "Medium", "Strict-Transport-Security header not present on HTTPS response",
                        "Header absent from the response.", why, fix)]
    match = re.search(r"max-age\s*=\s*\"?(\d+)", value, re.I)
    if not match:
        return [Finding("Security Header", name, "Low", "HSTS header has no valid max-age",
                        f"{name}: {trunc(value)}", why, fix)]
    max_age = int(match.group(1))
    if max_age == 0:
        return [Finding("Security Header", name, "Medium", "HSTS is explicitly disabled (max-age=0)",
                        f"{name}: {trunc(value)}", why, fix)]
    if max_age < HSTS_MIN_MAX_AGE:
        return [Finding("Security Header", name, "Low", f"HSTS max-age is short ({max_age} seconds)",
                        f"{name}: {trunc(value)}", why, fix)]
    extras = []
    if "includesubdomains" not in value.lower():
        extras.append("includeSubDomains not set")
    if "preload" not in value.lower():
        extras.append("preload not set")
    suffix = f" (optional hardening: {', '.join(extras)})" if extras else ""
    return [Finding("Security Header", name, "Informational", "HSTS present with a reasonable max-age" + suffix,
                    f"{name}: {trunc(value)}", why, "No action needed; consider includeSubDomains/preload if appropriate for all subdomains.")]


def check_x_content_type_options(snap: ResponseSnapshot):
    name = "X-Content-Type-Options"
    value = _header(snap, name)
    why = "'nosniff' stops browsers from guessing a resource's type, reducing MIME-confusion attacks."
    fix = "Send 'X-Content-Type-Options: nosniff' on all responses and set correct Content-Type values."
    if value is None:
        return [Finding("Security Header", name, "Low", "X-Content-Type-Options header not present", "Header absent from the response.", why, fix)]
    if value.strip().lower() != "nosniff":
        return [Finding("Security Header", name, "Low", "X-Content-Type-Options has an unexpected value", f"{name}: {trunc(value)}", why, fix)]
    return [Finding("Security Header", name, "Informational", "X-Content-Type-Options is set to nosniff", f"{name}: nosniff", why, "No action needed.")]


def check_x_frame_options(snap: ResponseSnapshot):
    name = "X-Frame-Options"
    value = _header(snap, name)
    csp = _header(snap, "Content-Security-Policy")
    has_frame_ancestors = bool(csp) and "frame-ancestors" in _csp_directives(csp)
    why = "Controlling who may frame a page mitigates clickjacking, where a page is hidden inside another site's frame to trick clicks."
    fix = "Prefer CSP 'frame-ancestors' (e.g. 'none' or 'self'); optionally also send 'X-Frame-Options: DENY' or 'SAMEORIGIN'."
    if value is None:
        if has_frame_ancestors:
            return [Finding("Security Header", name, "Informational", "X-Frame-Options absent, but CSP frame-ancestors is present",
                            "CSP frame-ancestors directive found.", why, "No action needed; frame-ancestors is the modern control.")]
        return [Finding("Security Header", name, "Low", "No clickjacking protection header found (X-Frame-Options / frame-ancestors)",
                        "Neither X-Frame-Options nor CSP frame-ancestors present.", why, fix)]
    if value.strip().upper() not in ("DENY", "SAMEORIGIN"):
        return [Finding("Security Header", name, "Low", "X-Frame-Options has an unrecognised or obsolete value",
                        f"{name}: {trunc(value)}",
                        why + " Values such as ALLOW-FROM are obsolete and ignored by modern browsers.", fix)]
    return [Finding("Security Header", name, "Informational", "X-Frame-Options is set", f"{name}: {trunc(value)}", why, "No action needed.")]


def check_referrer_policy(snap: ResponseSnapshot):
    name = "Referrer-Policy"
    value = _header(snap, name)
    why = "Referrer-Policy limits how much of the page URL is sent to other sites; URLs can contain sensitive identifiers."
    fix = "Send 'Referrer-Policy: strict-origin-when-cross-origin' (or stricter, such as 'no-referrer')."
    if value is None:
        return [Finding("Security Header", name, "Informational", "Referrer-Policy header not present",
                        "Header absent; modern browsers apply their own default (commonly strict-origin-when-cross-origin).", why, fix)]
    last_token = value.split(",")[-1].strip().lower()  # browsers use the last recognised token
    if last_token == "unsafe-url":
        return [Finding("Security Header", name, "Low", "Referrer-Policy sends full URLs to every destination",
                        f"{name}: {trunc(value)}", why, fix)]
    return [Finding("Security Header", name, "Informational", "Referrer-Policy is set", f"{name}: {trunc(value)}", why, "No action needed.")]


def check_permissions_policy(snap: ResponseSnapshot):
    name = "Permissions-Policy"
    value = _header(snap, name)
    why = "Permissions-Policy lets a site switch off powerful browser features (camera, microphone, geolocation) it does not use."
    fix = "Send e.g. 'Permissions-Policy: geolocation=(), camera=(), microphone=()' for features the site does not need."
    if value is None:
        return [Finding("Security Header", name, "Informational", "Permissions-Policy header not present", "Header absent from the response.", why, fix)]
    return [Finding("Security Header", name, "Informational", "Permissions-Policy is set", f"{name}: {trunc(value)}", why, "No action needed.")]


def check_security_headers(snap: ResponseSnapshot):
    findings = []
    for check in (check_csp, check_hsts, check_x_content_type_options,
                  check_x_frame_options, check_referrer_policy, check_permissions_policy):
        findings.extend(check(snap))
    return findings


print("Header checks defined:", [fn.__name__ for fn in (check_csp, check_hsts, check_x_content_type_options, check_x_frame_options, check_referrer_policy, check_permissions_policy)])

Header checks defined: ['check_csp', 'check_hsts', 'check_x_content_type_options', 'check_x_frame_options', 'check_referrer_policy', 'check_permissions_policy']

## 11. Cookie security flag checks
| Attribute | Purpose | Applicability caveats |
|---|---|---|
| **Secure** | Cookie is sent only over HTTPS | Only meaningful when the site uses HTTPS; this scan notes when the response itself arrived over HTTP |
| **HttpOnly** | Hides the cookie from JavaScript (limits theft via XSS) | Not every cookie *should* be HttpOnly — e.g. a preference or CSRF cookie read by front-end code may be intentionally script-readable |
| **SameSite** | Controls whether the cookie is sent on cross-site requests (`Strict`, `Lax`, `None`) | `None` requires `Secure`; many browsers default to `Lax` when absent |

**Privacy by design:** only the cookie *name* and its *attributes* are analysed. The cookie **value is discarded** at parse time and can never appear in a report. Only cookies set on the scanned response are seen — cookies set after login or on other pages are out of scope.

In [7]:
SESSION_LIKE = re.compile(r"(sess|sid|auth|token|jwt|login)", re.I)


def parse_set_cookie(raw: str) -> dict:
    """Parse one Set-Cookie line into name + lower-cased attributes. The cookie VALUE is discarded."""
    pieces = raw.split(";")
    first = pieces[0].strip()
    name = first.split("=", 1)[0].strip() if "=" in first else ""
    attrs = {}
    for piece in pieces[1:]:
        key, _, val = piece.strip().partition("=")
        if key:
            attrs[key.strip().lower()] = val.strip()
    return {"name": name or "(unnamed)", "attrs": attrs}


def check_cookies(snap: ResponseSnapshot):
    if not snap.set_cookies:
        return [Finding("Cookie", "Set-Cookie", "Informational", "No cookies were set on this response (cookie checks not applicable)",
                        "No Set-Cookie header in the scanned response.",
                        "Cookies set after login or on other pages are not visible to a single-request scan.",
                        "No action needed. Scan authenticated flows manually if they set cookies.")]
    over_https = snap.final_url.lower().startswith("https://")
    findings = []
    for raw in snap.set_cookies:
        cookie = parse_set_cookie(raw)
        name, attrs = cookie["name"], cookie["attrs"]
        session_like = bool(SESSION_LIKE.search(name))
        problems = 0
        label = f"Cookie '{name}'"
        hint = " The name looks session-related (heuristic)." if session_like else ""

        # --- Secure ---
        if "secure" not in attrs:
            problems += 1
            if over_https:
                sev = "Medium" if session_like else "Low"
                note = "The response arrived over HTTPS, so the cookie could be restricted to HTTPS."
            else:
                sev = "Low"
                note = "The response arrived over HTTP, so the Secure flag cannot be judged fully; it matters once the site uses HTTPS."
            findings.append(Finding("Cookie", f"{label} / Secure", sev, f"{label} lacks the Secure attribute",
                                    f"Attributes seen: {', '.join(sorted(attrs)) or 'none'}.",
                                    "Without Secure the browser may send the cookie over unencrypted HTTP, where it could be observed. " + note + hint,
                                    "Add the Secure attribute to cookies that should only travel over HTTPS (all session cookies)."))
        # --- HttpOnly ---
        if "httponly" not in attrs:
            problems += 1
            sev = "Medium" if session_like else "Low"
            findings.append(Finding("Cookie", f"{label} / HttpOnly", sev, f"{label} lacks the HttpOnly attribute",
                                    f"Attributes seen: {', '.join(sorted(attrs)) or 'none'}.",
                                    "Without HttpOnly, page scripts can read the cookie, which matters if script injection ever occurs. "
                                    "Some cookies (e.g. UI preferences, CSRF tokens) are intentionally readable by JavaScript." + hint,
                                    "Add HttpOnly to cookies that front-end code does not need to read, especially session and authentication cookies."))
        # --- SameSite ---
        samesite = attrs.get("samesite")
        if samesite is None:
            problems += 1
            findings.append(Finding("Cookie", f"{label} / SameSite", "Low", f"{label} has no SameSite attribute",
                                    f"Attributes seen: {', '.join(sorted(attrs)) or 'none'}.",
                                    "SameSite limits cross-site sending of cookies and helps against CSRF. Many modern browsers default to Lax, but behaviour varies.",
                                    "Set SameSite=Lax (or Strict) unless the cookie must be sent in cross-site contexts."))
        elif samesite.lower() == "none":
            if "secure" not in attrs:
                problems += 1
                findings.append(Finding("Cookie", f"{label} / SameSite", "Medium", f"{label} uses SameSite=None without Secure",
                                        "SameSite=None present, Secure absent.",
                                        "Browsers that follow the standard reject SameSite=None cookies that are not Secure, so the cookie may not work as intended.",
                                        "Add Secure to SameSite=None cookies, or use Lax/Strict if cross-site use is not required."))
            else:
                findings.append(Finding("Cookie", f"{label} / SameSite", "Informational", f"{label} is explicitly allowed in cross-site contexts (SameSite=None; Secure)",
                                        "SameSite=None; Secure present.",
                                        "This is legitimate for embedded or third-party use but widens exposure to cross-site requests.",
                                        "Confirm cross-site use is required; otherwise prefer Lax or Strict."))
        elif samesite.lower() not in ("lax", "strict"):
            problems += 1
            findings.append(Finding("Cookie", f"{label} / SameSite", "Low", f"{label} has an invalid SameSite value",
                                    f"SameSite={trunc(samesite, 30)}",
                                    "Browsers ignore invalid SameSite values and apply their default.",
                                    "Use SameSite=Lax, Strict or None (with Secure)."))
        if problems == 0:
            findings.append(Finding("Cookie", f"{label}", "Informational", f"{label}: Secure, HttpOnly and SameSite all present or intentionally set",
                                    f"Attributes seen: {', '.join(sorted(attrs))}.",
                                    "The three checked attributes look reasonable. Other attributes (Path, Domain, expiry) were not assessed.",
                                    "No action needed."))
    return findings


print("Cookie checks defined. Example parse (value is dropped):", parse_set_cookie("sessionid=SECRET123; Path=/; Secure; HttpOnly; SameSite=Lax"))

Cookie checks defined. Example parse (value is dropped): {'name': 'sessionid', 'attrs': {'path': '/', 'secure': '', 'httponly': '', 'samesite': 'Lax'}}

## 12. CORS configuration checks
Browsers enforce the *same-origin policy*; CORS headers let a server relax it for chosen origins. SecureScan inspects the response to the normal request and to **one additional `GET` that carries a harmless `Origin: https://cors-probe.invalid` header** (a reserved, non-resolving domain). It does **not** send preflight requests or attack payloads.

Interpretation is contextual:
- **No CORS headers** → default same-origin policy applies (Informational).
- **`Access-Control-Allow-Origin: *` alone** → any site may read this response *without credentials*. That is normal for public, non-sensitive resources, so it is **Informational**, not a vulnerability.
- **`*` with `Allow-Credentials: true`** → an invalid combination under the Fetch standard; browsers are expected to reject credentialed access. Reported as a **Medium** misconfiguration, *without claiming a browser would accept it* (not tested here).
- **Probe origin reflected back** → the server may echo any origin. Without credentials: **Low**. With `Allow-Credentials: true`: **High** *if confirmed* — any website could potentially read authenticated responses in a victim's browser. This tool does not verify real-world exploitability.
- **`null` origin** allowed → Medium (High with credentials), because sandboxed pages can send `Origin: null`.
- A **specific allow-listed origin** → Informational.

In [8]:
def check_cors(base: ResponseSnapshot, probe: Optional[ResponseSnapshot]):
    sources = [s for s in (probe, base) if s is not None and s.ok]
    probe_ok = probe is not None and probe.ok
    acao_probe = probe.headers.get("access-control-allow-origin") if probe_ok else None
    acao_base = base.headers.get("access-control-allow-origin")
    acao = acao_probe if acao_probe is not None else acao_base
    credentials = any(s.headers.get("access-control-allow-credentials", "").strip().lower() == "true" for s in sources)
    vary = (probe.headers.get("vary", "") if probe_ok else base.headers.get("vary", "")).lower()
    evidence = (f"Origin sent: {PROBE_ORIGIN if probe_ok else '(probe unavailable)'} | "
                f"Access-Control-Allow-Origin: {trunc(acao) if acao else '(absent)'} | "
                f"Access-Control-Allow-Credentials: {'true' if credentials else '(absent/false)'}")
    check, item = "CORS", "Access-Control-Allow-Origin"
    verify = " This tool did not test real browser behaviour or whether the endpoint returns sensitive data."

    if not acao and not credentials:
        return [Finding(check, item, "Informational", "No CORS headers returned (default same-origin policy applies)", evidence,
                        "Without CORS headers, browsers prevent other origins from reading the response.",
                        "No action needed unless cross-origin access is intended; then allow-list specific origins.")]
    if not acao:
        return [Finding(check, "Access-Control-Allow-Credentials", "Informational", "Allow-Credentials sent without an Allow-Origin header", evidence,
                        "Allow-Credentials has no effect without a matching Access-Control-Allow-Origin.",
                        "Remove the unused header or configure CORS deliberately.")]
    acao_clean = acao.strip()
    if acao_clean == "*":
        if credentials:
            return [Finding(check, item, "Medium", "Wildcard origin combined with Allow-Credentials: true", evidence,
                            "The Fetch standard does not allow '*' together with credentials, so browsers are expected to reject credentialed "
                            "cross-origin reads. The combination still signals a misconfiguration or a misunderstanding of CORS. "
                            "Browser acceptance was not tested here." + verify,
                            "Remove Allow-Credentials, or replace '*' with an explicit allow-list of trusted origins.")]
        return [Finding(check, item, "Informational", "Wildcard origin allowed (without credentials)", evidence,
                        "Any website can read this response from a browser, without cookies. That is normal for public, non-sensitive resources "
                        "and is not by itself a vulnerability.",
                        "Confirm the resource is public. Otherwise restrict Access-Control-Allow-Origin to trusted origins.")]
    if acao_clean.lower() == "null":
        sev = "High" if credentials else "Medium"
        return [Finding(check, item, sev, "'null' origin is allowed" + (" with credentials" if credentials else ""), evidence,
                        "Sandboxed iframes and some local files send 'Origin: null', so allowing it effectively allows untrusted contexts." + verify,
                        "Do not allow the 'null' origin; allow-list specific HTTPS origins.")]
    if probe_ok and acao_probe is not None and acao_probe.strip() == PROBE_ORIGIN:
        vary_note = "" if "origin" in vary else " The response also lacks 'Vary: Origin', which can cause caches to serve wrong CORS headers."
        if credentials:
            return [Finding(check, item, "High", "Arbitrary origin is reflected and credentials are allowed", evidence,
                            "The server echoed an origin it could not have pre-approved while allowing credentials. If this endpoint returns "
                            "authenticated or sensitive data, any website could potentially read it in a victim's browser. This is a potential "
                            "exposure to be confirmed manually, not a verified exploit." + vary_note + verify,
                            "Validate Origin against a fixed allow-list on the server; never reflect arbitrary origins when credentials are allowed; add 'Vary: Origin'.")]
        return [Finding(check, item, "Low", "Arbitrary origin is reflected (credentials not allowed)", evidence,
                        "The server echoes whatever origin is sent. Without credentials, other sites can only read what an anonymous visitor could already see." + vary_note + verify,
                        "Prefer an explicit allow-list of origins and add 'Vary: Origin' if the value changes per request.")]
    return [Finding(check, item, "Informational", "A specific origin is allow-listed" + (" (credentials allowed)" if credentials else ""), evidence,
                    "Allowing a specific, trusted origin is the recommended way to enable cross-origin access.",
                    "Keep the allow-list minimal and use HTTPS origins only.")]


print("check_cors defined.")

check_cors defined.

## 13. Putting it together: the scan workflow
`scan_target` is the single entry point:

1. Validate the URL (only http/https).
2. **Authorization gate** — non-loopback targets require `authorized=True`.
3. Fetch the target (with safe redirects). If it fails, return a clear *Informational* finding instead of crashing.
4. Run the HTTPS, header, cookie and CORS checks.
5. Return a `ScanResult`; `print_report` renders it and `export_reports` saves JSON/CSV.

In [9]:
def scan_target(url, authorized=False, allow_cross_host=False, timeout=None) -> ScanResult:
    """Scan one URL. Raises InvalidURLError for bad input and PermissionError if authorization is missing."""
    clean = validate_url(url)
    host = urlsplit(clean).hostname
    if not is_local_host(host) and not authorized:
        raise PermissionError(
            f"'{host}' is not a local address. Scan only systems you own or have written permission to test, "
            "then call scan_target(url, authorized=True).")

    result = ScanResult(target=clean, started_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"))
    primary = fetch_snapshot(clean, allow_cross_host=allow_cross_host, timeout=timeout)
    result.redirect_chain, result.notes = list(primary.redirect_chain), list(primary.notes)

    if not primary.ok:
        result.error = primary.error
        result.findings.append(Finding(
            "Connectivity", "Reachability", "Informational", "Target could not be scanned",
            f"{primary.error} (error type: {primary.error_type})",
            "A failed connection says nothing about the site's security configuration; the checks could not run.",
            "Check the URL spelling, your network/VPN/firewall, and that the server is running, then try again."))
        return result

    result.final_url, result.status_code = primary.final_url, primary.status_code

    if primary.redirect_chain:
        steps = " -> ".join(f"{code}" for code, _, _ in primary.redirect_chain)
        result.findings.append(Finding(
            "Redirects", "Redirect handling", "Informational", f"{len(primary.redirect_chain)} redirect(s) encountered",
            f"Statuses: {steps}. Final analysed URL: {trunc(primary.final_url, 120)}. " + " ".join(primary.notes),
            "Same-host redirects are followed (a leading 'www.' is treated as the same site); redirects to other hosts, other schemes or loops are not followed, so the scan never leaves the host you chose.",
            "No action needed. Use allow_cross_host=True only if you are authorized to test the destination too."))
    if primary.status_code >= 400:
        result.findings.append(Finding(
            "Response", "HTTP status", "Informational", f"Server returned status {primary.status_code}",
            f"Final URL: {trunc(primary.final_url, 120)}",
            "Error pages are often served by different components than normal pages and may carry different headers.",
            "Scan a representative page (e.g. the home page) for a more meaningful result."))

    https_findings, result.https_available = check_https(clean, primary, timeout=timeout)
    result.findings.extend(https_findings)
    result.findings.extend(check_security_headers(primary))
    result.findings.extend(check_cookies(primary))
    probe = fetch_snapshot(primary.final_url, extra_headers={"Origin": PROBE_ORIGIN}, timeout=timeout)
    result.findings.extend(check_cors(primary, probe))
    return result


def print_report(result: ScanResult, compact: bool = False) -> None:
    """Print a structured, readable summary. compact=True hides evidence/explanation/remediation."""
    line = "=" * 74
    print(line)
    print(f"{TOOL_NAME} v{TOOL_VERSION} - scan report")
    print(line)
    print(f"Target          : {result.target}")
    print(f"Scanned (UTC)   : {result.started_utc}")
    print(f"Final URL       : {result.final_url or '(none)'}")
    print(f"HTTP status     : {result.status_code if result.status_code is not None else '(none)'}")
    print(f"HTTPS available : {'yes' if result.https_available else 'no' if result.https_available is False else 'not assessed'}")
    if result.error:
        print(f"Scan error      : {result.error}")
    for note in result.notes:
        print(f"Note            : {note}")
    counts = result.counts()
    print("Findings        : " + " | ".join(f"{name} {counts[name]}" for name in reversed(SEVERITIES)))
    print(f"Highest severity: {result.highest()}")
    print("-" * 74)
    for f in sorted(result.findings, key=lambda x: -SEVERITY_RANK[x.severity]):
        print(f"\n[{f.severity.upper()}] {f.check} | {f.item}")
        print(f"  {f.title}")
        if not compact:
            for label, text in (("Evidence", f.evidence), ("Why it matters", f.explanation), ("Remediation", f.remediation)):
                print(textwrap.fill(f"{label}: {text}", width=88, initial_indent="    ", subsequent_indent="      "))
    print("\n" + textwrap.fill("NOTE: " + DISCLAIMER, width=88))
    print(line)


print("scan_target and print_report defined.")

scan_target and print_report defined.

## 14. Exporting JSON and CSV reports
Reports go to the `output/` directory (created automatically).

- **JSON** — full structure: tool info, disclaimer, scan metadata, summary counts, and every finding.
- **CSV** — one row per finding, convenient for spreadsheets.

Security details: the CSV writer neutralises **formula injection** (cells starting with `=`, `+`, `-`, `@` get a leading apostrophe) because evidence text originates from remote servers; evidence is truncated; and cookie values and response bodies are never present in the data.

In [10]:
CSV_COLUMNS = ["target", "severity", "check", "item", "title", "evidence", "explanation", "remediation"]


def csv_safe(value) -> str:
    """Prevent spreadsheet formula injection in CSV cells."""
    text = str(value)
    return "'" + text if text[:1] in ("=", "+", "-", "@", "\t", "\r") else text


def export_reports(result: ScanResult, out_dir=OUTPUT_DIR) -> dict:
    """Write JSON and CSV reports; return their paths."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    parts = urlsplit(result.target)
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{parts.hostname}_{parts.port or ''}{parts.path}").strip("_")[:60] or "target"
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    base = out_dir / f"securescan_{slug}_{stamp}"

    payload = {
        "tool": {"name": TOOL_NAME, "version": TOOL_VERSION},
        "disclaimer": DISCLAIMER,
        "scan": {
            "target": result.target, "started_utc": result.started_utc, "final_url": result.final_url,
            "status_code": result.status_code, "https_available": result.https_available,
            "redirects": [{"status": s, "from": a, "to": b} for s, a, b in result.redirect_chain],
            "notes": result.notes, "error": result.error,
        },
        "summary": {"counts": result.counts(), "highest_severity": result.highest(), "total_findings": len(result.findings)},
        "findings": [asdict(f) for f in sorted(result.findings, key=lambda x: -SEVERITY_RANK[x.severity])],
    }
    json_path = Path(str(base) + ".json")  # not with_suffix(): host names contain dots
    json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    csv_path = Path(str(base) + ".csv")
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(CSV_COLUMNS)
        for f in sorted(result.findings, key=lambda x: -SEVERITY_RANK[x.severity]):
            writer.writerow([csv_safe(v) for v in (result.target, f.severity, f.check, f.item, f.title, f.evidence, f.explanation, f.remediation)])
    return {"json": json_path, "csv": csv_path}


print("export_reports defined. Output directory:", OUTPUT_DIR.resolve())

export_reports defined. Output directory: <notebook folder>/output

## 15. Safe demonstration: a local test server
To demonstrate the scanner **without touching anyone else's website**, the notebook starts a tiny HTTP server on `127.0.0.1` (your own machine, random free port) in a background thread. It deliberately serves different configurations on different paths:

| Path | Purpose |
|---|---|
| `/insecure` | No security headers; session-like cookie with no flags; origin reflected with credentials |
| `/hardened` | Strong headers, protected cookie, a specific allow-listed CORS origin |
| `/mixed` | Weak CSP, obsolete X-Frame-Options, `unsafe-url` referrer, mixed cookies, wildcard CORS |
| `/wildcard-creds` | `*` with `Allow-Credentials: true` |
| `/redirect`, `/redirect-external`, `/redirect-loop` | Same-host redirect, cross-host redirect (must *not* be followed), loop |
| `/slow` | Responds after 3 s — used to demonstrate timeout handling |

The cookie values and headers below are fake demo data.

In [11]:
class DemoHandler(BaseHTTPRequestHandler):
    """Serves fixed, fake responses so every check can be demonstrated safely offline."""
    server_version = "DemoServer"
    sys_version = ""

    def log_message(self, *args):  # keep the notebook output clean
        pass

    def _send(self, status=200, headers=None, cookies=None, body=b"SecureScan demo page"):
        self.send_response(status)
        for key, value in (headers or {}).items():
            self.send_header(key, value)
        for cookie in cookies or []:
            self.send_header("Set-Cookie", cookie)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        try:
            if self.command != "HEAD":
                self.wfile.write(body)
        except (BrokenPipeError, ConnectionResetError):
            pass  # the client gave up (e.g. the timeout demo); nothing to clean up

    def do_GET(self):
        path = self.path.split("?")[0]
        origin = self.headers.get("Origin")
        if path == "/insecure":
            headers = {}
            if origin:  # deliberately reflects any origin AND allows credentials (demo of a risky config)
                headers.update({"Access-Control-Allow-Origin": origin, "Access-Control-Allow-Credentials": "true"})
            self._send(headers=headers, cookies=["sessionid=demo-fake-value; Path=/", "theme=dark; Path=/"])
        elif path == "/hardened":
            self._send(headers={
                "Content-Security-Policy": "default-src 'self'; script-src 'self'; object-src 'none'; frame-ancestors 'none'; base-uri 'self'",
                "Strict-Transport-Security": "max-age=31536000; includeSubDomains",
                "X-Content-Type-Options": "nosniff",
                "X-Frame-Options": "DENY",
                "Referrer-Policy": "strict-origin-when-cross-origin",
                "Permissions-Policy": "geolocation=(), camera=(), microphone=()",
                "Access-Control-Allow-Origin": "https://trusted.example",
            }, cookies=["sessionid=demo-fake-value; Path=/; Secure; HttpOnly; SameSite=Strict"])
        elif path == "/mixed":
            self._send(headers={
                "Content-Security-Policy": "default-src 'self'; script-src 'self' 'unsafe-inline'",
                "X-Content-Type-Options": "nosniff",
                "X-Frame-Options": "ALLOW-FROM https://partner.example",
                "Referrer-Policy": "unsafe-url",
                "Access-Control-Allow-Origin": "*",
            }, cookies=["prefs=compact; Path=/; SameSite=None", "auth_token=demo-fake-value; Path=/; Secure"])
        elif path == "/wildcard-creds":
            self._send(headers={"Access-Control-Allow-Origin": "*", "Access-Control-Allow-Credentials": "true"})
        elif path == "/redirect":
            self._send(302, headers={"Location": "/hardened"}, body=b"")
        elif path == "/redirect-external":
            self._send(302, headers={"Location": "http://example.org/"}, body=b"")
        elif path == "/redirect-loop":
            self._send(302, headers={"Location": "/redirect-loop"}, body=b"")
        elif path == "/slow":
            time.sleep(3)
            self._send()
        else:
            self._send(404, body=b"Not found")


def start_demo_server():
    """Start the demo server on a random free loopback port; return (server, base_url)."""
    server = ThreadingHTTPServer(("127.0.0.1", 0), DemoHandler)
    server.daemon_threads = True
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"


def stop_demo_server(server) -> None:
    server.shutdown()
    server.server_close()


print("Demo server code defined (not started yet).")

Demo server code defined (not started yet).

## 16. Demonstration scans
First we scan the deliberately **insecure** demo page and print the full report. Then the other pages are scanned and shown in compact form.

In [12]:
demo_server, BASE = start_demo_server()
print("Demo server listening on", BASE, "(loopback only)\n")

demo_results = {}
demo_results["Insecure demo page"] = scan_target(BASE + "/insecure")
print_report(demo_results["Insecure demo page"])

Demo server listening on http://127.0.0.1:46165 (loopback only)

SecureScan v1.0.0 - scan report
Target          : http://127.0.0.1:46165/insecure
Scanned (UTC)   : 2026-10-25T02:38:06+00:00
Final URL       : http://127.0.0.1:46165/insecure
HTTP status     : 200
HTTPS available : not assessed
Findings        : High 1 | Medium 2 | Low 7 | Informational 4
Highest severity: High
--------------------------------------------------------------------------

[HIGH] CORS | Access-Control-Allow-Origin
  Arbitrary origin is reflected and credentials are allowed
    Evidence: Origin sent: https://cors-probe.invalid | Access-Control-Allow-Origin:
      https://cors-probe.invalid | Access-Control-Allow-Credentials: true
    Why it matters: The server echoed an origin it could not have pre-approved while
      allowing credentials. If this endpoint returns authenticated or sensitive data,
      any website could potentially read it in a victim's browser. This is a potential
      exposure to be confi

In [13]:
for label, path in [("Hardened demo page", "/hardened"), ("Mixed demo page", "/mixed"), ("Wildcard + credentials page", "/wildcard-creds")]:
    demo_results[label] = scan_target(BASE + path)
    print(f"\n##### {label} #####")
    print_report(demo_results[label], compact=True)


##### Hardened demo page #####
SecureScan v1.0.0 - scan report
Target          : http://127.0.0.1:46165/hardened
Scanned (UTC)   : 2026-10-25T02:38:06+00:00
Final URL       : http://127.0.0.1:46165/hardened
HTTP status     : 200
HTTPS available : not assessed
Findings        : High 0 | Medium 0 | Low 0 | Informational 9
Highest severity: Informational
--------------------------------------------------------------------------

[INFORMATIONAL] HTTPS | Transport
  Local test server uses plain HTTP (HTTPS check skipped)

[INFORMATIONAL] Security Header | Content-Security-Policy
  CSP present; no obvious weaknesses found by simple checks

[INFORMATIONAL] Security Header | Strict-Transport-Security
  HSTS not applicable: response was served over HTTP

[INFORMATIONAL] Security Header | X-Content-Type-Options
  X-Content-Type-Options is set to nosniff

[INFORMATIONAL] Security Header | X-Frame-Options
  X-Frame-Options is set

[INFORMATIONAL] Security Header | Referrer-Policy
  Referrer-Polic

### Redirect handling, unreachable targets and timeouts
These scans show that the scanner **does not crash** on awkward situations and explains what happened:
- a same-host redirect (followed),
- a redirect to a *different* host (deliberately **not** followed — and no request is ever sent there),
- a redirect loop,
- a closed port (connection refused),
- a name that cannot be resolved (`.invalid` is a reserved TLD guaranteed never to exist),
- a slow server with a short timeout.

In [14]:
import socket as _socket

edge_cases = {}
edge_cases["Same-host redirect"] = scan_target(BASE + "/redirect")
edge_cases["Cross-host redirect"] = scan_target(BASE + "/redirect-external")
edge_cases["Redirect loop"] = scan_target(BASE + "/redirect-loop")

with _socket.socket() as s:            # find a port that is certainly closed
    s.bind(("127.0.0.1", 0))
    closed_port = s.getsockname()[1]
edge_cases["Closed port"] = scan_target(f"http://127.0.0.1:{closed_port}/")
edge_cases["Unresolvable name"] = scan_target("http://no-such-host.invalid/", authorized=True)
edge_cases["Slow server (timeout 2s/1s)"] = scan_target(BASE + "/slow", timeout=(2, 1))

for label, res in edge_cases.items():
    note = res.error or (res.notes[0] if res.notes else f"final URL {res.final_url}")
    print(f"{label:30} -> highest: {res.highest():13} | {trunc(note, 95)}")

Same-host redirect             -> highest: Informational | final URL http://127.0.0.1:46165/hardened
Cross-host redirect            -> highest: Medium        | Redirect to a different host (example.org) was NOT followed; results describe the redirect r...
Redirect loop                  -> highest: Medium        | A redirect loop was detected; following stopped.
Closed port                    -> highest: Informational | Connection refused (nothing is listening on that host/port).
Unresolvable name              -> highest: Informational | DNS lookup failed (the host name could not be resolved).
Slow server (timeout 2s/1s)    -> highest: Informational | The connection or response timed out.

## 17. Export the demonstration reports
Every demonstration scan is written to `output/` as JSON and CSV.

In [15]:
exported = {}
for label, res in {**demo_results, **edge_cases}.items():
    exported[label] = export_reports(res)

for label, paths in exported.items():
    print(f"{label:30} {paths['json'].name}  +  {paths['csv'].name}")
print("\nTotal files written:", sum(len(p) for p in exported.values()), "in", OUTPUT_DIR.resolve())

# Show the start of one JSON report to illustrate its structure.
sample_json = json.loads(exported["Insecure demo page"]["json"].read_text(encoding="utf-8"))
print("\nJSON keys:", list(sample_json))
print("Summary  :", sample_json["summary"])
print("First finding:", {k: trunc(v, 70) for k, v in sample_json["findings"][0].items()})

Insecure demo page             securescan_127.0.0.1_46165_insecure_20261025_023807.json  +  securescan_127.0.0.1_46165_insecure_20261025_023807.csv
Hardened demo page             securescan_127.0.0.1_46165_hardened_20261025_023807.json  +  securescan_127.0.0.1_46165_hardened_20261025_023807.csv
Mixed demo page                securescan_127.0.0.1_46165_mixed_20261025_023807.json  +  securescan_127.0.0.1_46165_mixed_20261025_023807.csv
Wildcard + credentials page    securescan_127.0.0.1_46165_wildcard-creds_20261025_023807.json  +  securescan_127.0.0.1_46165_wildcard-creds_20261025_023807.csv
Same-host redirect             securescan_127.0.0.1_46165_redirect_20261025_023807.json  +  securescan_127.0.0.1_46165_redirect_20261025_023807.csv
Cross-host redirect            securescan_127.0.0.1_46165_redirect-external_20261025_023807.json  +  securescan_127.0.0.1_46165_redirect-external_20261025_023807.csv
Redirect loop                  securescan_127.0.0.1_46165_redirect-loop_20261025_023807.

## 18. Automated tests
A lightweight test runner (no extra dependency) checks the logic using **mock responses** and the **local demo server**:
URL validation, every header/cookie/CORS rule, privacy (cookie values never leak), redirect safety, error handling (refused, DNS, timeout), the authorization gate, and the JSON/CSV exports.
Each test prints PASS or FAIL; the cell raises an error if any test fails.

In [16]:
TESTS = []


def test(fn):
    TESTS.append(fn)
    return fn


def mock(headers=None, cookies=None, url="https://example.test/"):
    """Build a ResponseSnapshot from plain data (no network)."""
    return ResponseSnapshot(requested_url=url, final_url=url, status_code=200,
                            headers={k.lower(): v for k, v in (headers or {}).items()}, set_cookies=list(cookies or []))


def sev_of(findings, text):
    """Severity of the first finding whose item or title contains `text`."""
    for f in findings:
        if text.lower() in (f.item + " " + f.title).lower():
            return f.severity
    return None


@test
def t01_url_validation_accepts_good_urls():
    assert validate_url("example.com") == "https://example.com"
    assert validate_url("http://127.0.0.1:8000/a#frag") == "http://127.0.0.1:8000/a"
    assert validate_url("  https://Example.com/path?q=1 ") == "https://Example.com/path?q=1"


@test
def t02_url_validation_rejects_bad_urls():
    for bad in ["", "   ", None, "ftp://example.com", "file:///etc/passwd", "javascript:alert(1)",
                "https://user:pw@example.com", "http://exa mple.com", "http://example.com:99999",
                "http://-bad-.com", "https://" + "a" * 3000 + ".com"]:
        try:
            validate_url(bad)
        except InvalidURLError:
            continue
        raise AssertionError(f"should have been rejected: {str(bad)[:40]!r}")


@test
def t03_csp_rules():
    assert sev_of(check_csp(mock()), "Content-Security-Policy") == "Medium"
    assert sev_of(check_csp(mock({"Content-Security-Policy-Report-Only": "default-src 'self'"})), "report-only") == "Low"
    assert check_csp(mock({"Content-Security-Policy": "default-src 'self'; script-src 'self' 'unsafe-inline'"}))[0].severity == "Low"
    assert check_csp(mock({"Content-Security-Policy": "script-src *"}))[0].severity == "Medium"
    assert check_csp(mock({"Content-Security-Policy": "frame-ancestors 'none'"}))[0].severity == "Low"
    assert check_csp(mock({"Content-Security-Policy": "default-src 'self'; script-src 'nonce-abc' 'unsafe-inline'"}))[0].severity == "Informational"
    assert check_csp(mock({"Content-Security-Policy": "default-src 'self'; object-src 'none'"}))[0].severity == "Informational"


@test
def t04_hsts_rules():
    assert check_hsts(mock())[0].severity == "Medium"                                            # HTTPS, missing
    assert check_hsts(mock(url="http://example.test/"))[0].severity == "Informational"          # HTTP: not applicable
    assert check_hsts(mock({"Strict-Transport-Security": "max-age=60"}))[0].severity == "Low"
    assert check_hsts(mock({"Strict-Transport-Security": "max-age=0"}))[0].severity == "Medium"
    assert check_hsts(mock({"Strict-Transport-Security": "includeSubDomains"}))[0].severity == "Low"
    assert check_hsts(mock({"Strict-Transport-Security": "max-age=31536000; includeSubDomains; preload"}))[0].severity == "Informational"


@test
def t05_other_header_rules():
    assert check_x_content_type_options(mock())[0].severity == "Low"
    assert check_x_content_type_options(mock({"X-Content-Type-Options": "sniff"}))[0].severity == "Low"
    assert check_x_content_type_options(mock({"X-Content-Type-Options": "NoSniff"}))[0].severity == "Informational"
    assert check_x_frame_options(mock())[0].severity == "Low"
    assert check_x_frame_options(mock({"Content-Security-Policy": "frame-ancestors 'self'"}))[0].severity == "Informational"
    assert check_x_frame_options(mock({"X-Frame-Options": "ALLOW-FROM https://a.example"}))[0].severity == "Low"
    assert check_x_frame_options(mock({"X-Frame-Options": "deny"}))[0].severity == "Informational"
    assert check_referrer_policy(mock())[0].severity == "Informational"
    assert check_referrer_policy(mock({"Referrer-Policy": "no-referrer, unsafe-url"}))[0].severity == "Low"
    assert check_referrer_policy(mock({"Referrer-Policy": "no-referrer"}))[0].severity == "Informational"
    assert check_permissions_policy(mock())[0].severity == "Informational"


@test
def t06_cookie_rules():
    bare = check_cookies(mock(cookies=["sessionid=secretvalue123; Path=/"]))
    assert sev_of(bare, "Secure") == "Medium" and sev_of(bare, "HttpOnly") == "Medium" and sev_of(bare, "SameSite") == "Low"
    plain = check_cookies(mock(cookies=["theme=dark"]))
    assert sev_of(plain, "Secure") == "Low" and sev_of(plain, "HttpOnly") == "Low"           # non-session names are Low
    http_page = check_cookies(mock(cookies=["sessionid=x; HttpOnly; SameSite=Lax"], url="http://example.test/"))
    assert sev_of(http_page, "Secure") == "Low"                                              # Secure not judgeable over HTTP
    good = check_cookies(mock(cookies=["sessionid=x; Secure; HttpOnly; SameSite=Strict"]))
    assert [f.severity for f in good] == ["Informational"]
    assert sev_of(check_cookies(mock(cookies=["a=1; Secure; HttpOnly; SameSite=None"])), "SameSite") == "Informational"
    assert sev_of(check_cookies(mock(cookies=["a=1; HttpOnly; SameSite=None"])), "SameSite") == "Medium"
    assert sev_of(check_cookies(mock(cookies=["a=1; Secure; HttpOnly; SameSite=Banana"])), "SameSite") == "Low"
    assert check_cookies(mock())[0].severity == "Informational"                              # no cookies
    assert parse_set_cookie("x=SECRET; Path=/; Secure") == {"name": "x", "attrs": {"path": "/", "secure": ""}}


@test
def t07_cookie_values_never_leak():
    findings = check_cookies(mock(cookies=["sessionid=secretvalue123; Path=/", "auth=topsecret456"]))
    blob = json.dumps([asdict(f) for f in findings])
    assert "secretvalue123" not in blob and "topsecret456" not in blob


@test
def t08_cors_rules():
    base = mock()
    def probe_with(headers):
        return mock(headers)
    assert check_cors(base, probe_with({}))[0].severity == "Informational"
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": "*"}))[0].severity == "Informational"
    wc = check_cors(base, probe_with({"Access-Control-Allow-Origin": "*", "Access-Control-Allow-Credentials": "true"}))[0]
    assert wc.severity == "Medium" and "not tested" in wc.explanation.lower()               # no claim of browser acceptance
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": PROBE_ORIGIN}))[0].severity == "Low"
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": PROBE_ORIGIN, "Access-Control-Allow-Credentials": "true"}))[0].severity == "High"
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": "null"}))[0].severity == "Medium"
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": "null", "Access-Control-Allow-Credentials": "true"}))[0].severity == "High"
    assert check_cors(base, probe_with({"Access-Control-Allow-Origin": "https://trusted.example"}))[0].severity == "Informational"
    assert check_cors(base, None)[0].severity == "Informational"                             # probe failed: still no crash


@test
def t09_every_finding_has_remediation_and_valid_severity():
    samples = []
    samples += check_security_headers(mock())
    samples += check_security_headers(mock({"Content-Security-Policy": "script-src *", "Strict-Transport-Security": "max-age=1", "X-Frame-Options": "x", "Referrer-Policy": "unsafe-url"}))
    samples += check_cookies(mock(cookies=["sessionid=x", "a=1; SameSite=None", "b=1; Secure; HttpOnly; SameSite=Lax"]))
    for hdr in ({}, {"Access-Control-Allow-Origin": "*"}, {"Access-Control-Allow-Origin": PROBE_ORIGIN, "Access-Control-Allow-Credentials": "true"}):
        samples += check_cors(mock(), mock(hdr))
    assert len(samples) > 20
    for f in samples:
        assert f.remediation.strip() and f.explanation.strip() and f.evidence.strip()
        assert f.severity in SEVERITIES


@test
def t10_local_demo_server_scans():
    server, base = start_demo_server()
    try:
        bad = scan_target(base + "/insecure")
        assert bad.highest() == "High" and sev_of(bad.findings, "Arbitrary origin") == "High"
        assert sev_of(bad.findings, "'sessionid' / HttpOnly") == "Medium"
        good = scan_target(base + "/hardened")
        assert good.highest() == "Informational", good.counts()
        assert good.status_code == 200 and good.https_available is None
        assert scan_target(base + "/wildcard-creds").highest() == "Medium"
        mixed = scan_target(base + "/mixed")
        assert sev_of(mixed.findings, "CSP present but weak") == "Low" and sev_of(mixed.findings, "'auth_token' / HttpOnly") == "Medium"
        assert scan_target(base + "/nope").status_code == 404
    finally:
        stop_demo_server(server)


@test
def t11_redirect_safety():
    server, base = start_demo_server()
    try:
        same = fetch_snapshot(base + "/redirect")
        assert same.final_url.endswith("/hardened") and len(same.redirect_chain) == 1 and same.status_code == 200
        ext = fetch_snapshot(base + "/redirect-external")
        assert ext.final_url.endswith("/redirect-external") and ext.status_code == 302     # NOT followed
        assert any("different host" in n for n in ext.notes)
        loop = fetch_snapshot(base + "/redirect-loop")
        assert any("loop" in n for n in loop.notes) and loop.ok
    finally:
        stop_demo_server(server)


@test
def t12_network_failures_do_not_crash():
    server, base = start_demo_server()
    try:
        with _socket.socket() as s:
            s.bind(("127.0.0.1", 0))
            port = s.getsockname()[1]
        refused = scan_target(f"http://127.0.0.1:{port}/")
        assert refused.error and refused.findings[0].title == "Target could not be scanned"
        assert fetch_snapshot(f"http://127.0.0.1:{port}/").error_type == "refused"
        dns = scan_target("http://no-such-host.invalid/", authorized=True)
        assert dns.error and dns.findings[0].title == "Target could not be scanned"
        slow = fetch_snapshot(base + "/slow", timeout=(2, 1))
        assert slow.error_type == "timeout"
    finally:
        stop_demo_server(server)


@test
def t13_authorization_gate_and_scheme_gate():
    for target in ("http://example.org", "https://192.0.2.1"):
        try:
            scan_target(target)           # must refuse BEFORE any network traffic
        except PermissionError:
            continue
        raise AssertionError("non-local target scanned without authorization")
    try:
        scan_target("ftp://127.0.0.1/")
    except InvalidURLError:
        pass
    else:
        raise AssertionError("ftp scheme accepted")


@test
def t14_https_logic_with_mocks():
    # HTTPS response reached directly: no extra HTTPS probe needed, HTTP probe to a closed place is not required to assert this part
    primary = mock(url="https://example.test:8443/")           # non-standard port: no counterpart probes, no network
    findings, ok = check_https("https://example.test:8443/", primary)
    assert ok is True and findings[0].severity == "Informational"
    primary_http = mock(url="http://example.test:8080/")
    findings, ok = check_https("http://example.test:8080/", primary_http)
    assert ok is None and findings[0].severity == "Medium"
    findings, ok = check_https("http://127.0.0.1:8000/", mock(url="http://127.0.0.1:8000/"))
    assert ok is None and findings[0].severity == "Informational"


@test
def t15_exports_json_and_csv():
    res = ScanResult(target="http://127.0.0.1:9/x", started_utc="2026-01-01T00:00:00+00:00", final_url="http://127.0.0.1:9/x", status_code=200)
    res.findings = [
        Finding("Security Header", "X", "Low", "=HYPERLINK(\"http://evil\")", "+evidence", "-why", "@fix"),
        Finding("CORS", "Y", "High", "plain, with comma", "e", "w", "r"),
    ]
    with tempfile.TemporaryDirectory() as tmp:
        paths = export_reports(res, tmp)
        data = json.loads(paths["json"].read_text(encoding="utf-8"))
        assert data["summary"]["counts"]["High"] == 1 and data["summary"]["total_findings"] == 2
        assert data["findings"][0]["severity"] == "High"          # sorted most severe first
        assert "disclaimer" in data and data["tool"]["name"] == TOOL_NAME
        assert paths["json"].name.startswith("securescan_127.0.0.1_9_x_") and paths["csv"].suffix == ".csv"
        with paths["csv"].open(newline="", encoding="utf-8") as handle:
            rows = list(csv.reader(handle))
        assert rows[0] == CSV_COLUMNS and len(rows) == 3
        formula_row = next(r for r in rows if r[1] == "Low")
        assert formula_row[4].startswith("'=") and formula_row[5].startswith("'+") and formula_row[6].startswith("'-") and formula_row[7].startswith("'@")
        assert any(r[4] == "plain, with comma" for r in rows)     # commas handled by CSV quoting


def run_tests():
    passed, failed = 0, []
    for fn in TESTS:
        try:
            fn()
            passed += 1
            print(f"PASS  {fn.__name__}")
        except AssertionError as exc:
            failed.append(fn.__name__)
            print(f"FAIL  {fn.__name__}: {exc}")
        except Exception as exc:  # unexpected error inside a test
            failed.append(fn.__name__)
            print(f"ERROR {fn.__name__}: {type(exc).__name__}: {exc}")
    print(f"\n{passed} passed, {len(failed)} failed, {len(TESTS)} total")
    return failed


failed_tests = run_tests()
assert not failed_tests, f"Tests failed: {failed_tests}"

PASS  t01_url_validation_accepts_good_urls
PASS  t02_url_validation_rejects_bad_urls
PASS  t03_csp_rules
PASS  t04_hsts_rules
PASS  t05_other_header_rules
PASS  t06_cookie_rules
PASS  t07_cookie_values_never_leak
PASS  t08_cors_rules
PASS  t09_every_finding_has_remediation_and_valid_severity
PASS  t10_local_demo_server_scans
PASS  t11_redirect_safety
PASS  t12_network_failures_do_not_crash
PASS  t13_authorization_gate_and_scheme_gate
PASS  t14_https_logic_with_mocks
PASS  t15_exports_json_and_csv

15 passed, 0 failed, 15 total

## 19. Optional: scan a URL you enter yourself
Set `RUN_INTERACTIVE_SCAN = True` and re-run this cell to scan a URL of your choice. For **non-local** targets you will be asked to confirm that you are **authorized** to test them. Leave it `False` for a clean "Run All".

In [17]:
RUN_INTERACTIVE_SCAN = False

if RUN_INTERACTIVE_SCAN:
    raw_url = input("Enter the URL to scan (e.g. http://127.0.0.1:8000 or https://your-own-site.example): ")
    try:
        clean_url = validate_url(raw_url)
        authorized = is_local_host(urlsplit(clean_url).hostname)
        if not authorized:
            answer = input("Do you own this system or have WRITTEN permission to test it? (yes/no): ")
            authorized = answer.strip().lower() == "yes"
        user_result = scan_target(clean_url, authorized=authorized)
        print_report(user_result)
        print("Reports saved:", {k: str(v) for k, v in export_reports(user_result).items()})
    except (InvalidURLError, PermissionError) as err:
        print("Not scanned:", err)
else:
    print("Interactive scan is switched off. Set RUN_INTERACTIVE_SCAN = True above to scan a URL you are authorized to test.")

Interactive scan is switched off. Set RUN_INTERACTIVE_SCAN = True above to scan a URL you are authorized to test.

## 20. Final results
The table below is **computed from the scans that just ran** in this notebook (nothing is typed in by hand).

In [18]:
rows = [(label, res) for label, res in {**demo_results, **edge_cases}.items()]
print(f"{'Scenario':30} {'Status':>6} {'High':>5} {'Med':>4} {'Low':>4} {'Info':>5}  Highest")
print("-" * 78)
for label, res in rows:
    c = res.counts()
    status = res.status_code if res.status_code is not None else "-"
    print(f"{label:30} {str(status):>6} {c['High']:>5} {c['Medium']:>4} {c['Low']:>4} {c['Informational']:>5}  {res.highest()}")

stop_demo_server(demo_server)
print("\nDemo server stopped. Reports are in:", OUTPUT_DIR.resolve())

Scenario                       Status  High  Med  Low  Info  Highest
------------------------------------------------------------------------------
Insecure demo page                200     1    2    7     4  High
Hardened demo page                200     0    0    0     9  Informational
Mixed demo page                   200     0    2    6     5  Medium
Wildcard + credentials page       200     0    2    2     5  Medium
Same-host redirect                200     0    0    0    10  Informational
Cross-host redirect               302     0    1    2     7  Medium
Redirect loop                     302     0    1    2     7  Medium
Closed port                         -     0    0    0     1  Informational
Unresolvable name                   -     0    0    0     1  Informational
Slow server (timeout 2s/1s)         -     0    0    0     1  Informational

Demo server stopped. Reports are in: <notebook folder>/output

## 21. Conclusion

**What was achieved.** SecureScan checks HTTPS availability, six HTTP security headers, cookie attributes and CORS behaviour; classifies each observation as Informational, Low, Medium or High using documented rules; explains why each finding matters and how to remediate it; and exports JSON and CSV reports. The demonstration shows the expected contrast: the deliberately insecure local page produces High and Medium findings, while the hardened page produces only Informational ones. Automated tests confirm the rules, the privacy guarantees (no cookie values or bodies in output), the redirect and authorization safeguards, graceful failure handling and the exports.

**How to read results responsibly.** Findings are *preliminary configuration observations*. A missing header is a missing defensive control — not proof that a site is vulnerable. The CORS "High" cases describe a *potential* exposure that needs manual confirmation, and the tool does not test real browser behaviour or exploitability.

**Limitations.**
- Only the response to the scanned URL is analysed; authenticated areas, other pages and API routes are not crawled.
- No preflight (`OPTIONS`) CORS testing, no TLS protocol/cipher analysis, no certificate-expiry reporting.
- Header checks are rule-of-thumb; CSP quality in particular needs expert review. Cookie "session-like" detection is a name-based heuristic.
- Live-internet behaviour (real certificates, CDNs, proxies) was not part of the automated tests, which use mock responses and a local server.

**Future enhancements.** Multi-page scanning of an allow-listed set of paths, preflight CORS checks, TLS version and certificate-expiry checks, an HTML report, a command-line interface, and a configurable policy file for severity rules.

**References**
- OWASP Secure Headers Project — https://owasp.org/www-project-secure-headers/
- OWASP Cheat Sheet Series: HTTP Headers — https://cheatsheetseries.owasp.org/cheatsheets/HTTP_Headers_Cheat_Sheet.html
- OWASP Cheat Sheet Series: Session Management (cookie attributes) — https://cheatsheetseries.owasp.org/cheatsheets/Session_Management_Cheat_Sheet.html
- OWASP Web Security Testing Guide: Testing CORS — https://owasp.org/www-project-web-security-testing-guide/latest/4-Web_Application_Security_Testing/11-Client-side_Testing/07-Testing_Cross_Origin_Resource_Sharing
- MDN: Content-Security-Policy — https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Content-Security-Policy
- MDN: Strict-Transport-Security — https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Strict-Transport-Security
- MDN: Set-Cookie — https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Set-Cookie
- MDN: Cross-Origin Resource Sharing (CORS) — https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/CORS

*Reminder: use SecureScan only on systems you own or are explicitly authorized to test.*